Investigated settings - for selection of region for MyCriteria

Done things 
    EEG
        LDA
        SVC
        RandomForest
    MEG
        LDA
        SVC
        RandomForest
        
    EEG+MEG
        RandomForest
        SVC
        LDA


Observations on spectrograms(see Visualization notebook):  <br>
    1. in every case (varying classifiers) Not personalized is better (or equivalent) to Personalized <br>
    2. For both SVM and RandomForest, NROIs Performance increase with NROIs, but starts to saturate already at 10 ROIs. For LDA can vary <br>
    3. SVM and RandomForest better than LDA in EEG and MEG alone. Improvement present but lower when using EEG concat MEG <br>
    4. both for SVM and RandomForest, EEG concat MEG perform as the best between MEG and EEG alone. For LDA, it can be better than the best of the 2 <br>

## Brainstormings - Idee

Todo:
- Fai classificazione con tutte le regioni; a posteriori di SVM controlla quali sono le più significative (T test su differenza tra MI e Rest)
- Prova test statitsico differente da T test per la scelta delle regioni e.g. Energy Distance Test / k-sample E-test / T test in corrispondenza solo della frq che ha il max diff Re-MI
- verifica come si comporta un MLP per feature estratte a mano (per avere una stima di che succederà con DeepLearning. In realtà non é detto che debba usare come calssificatore dopo le features extraction un MLP.... potrei usare RandomForest?? vediamo...)

Incontro 04/12/2025:
- Per utilizzare Deep Learning, potrebbe essere utile selezionare solo alcune regioni!
- Criterio per selezione regioni:
    ◊ investiga meglio cosa succede per il nostro criterio: "cross validation of region selection", se la selezione é significativa, pero ogni soggetto, anche al variare dei trials (split test train kfold) devo ottenere stessa selezione di regioni!
    ◊ DEEP Learning: trovare un buon criterio per fare in modo di selezionare regioni significative e semplificare il lavoro all'autoencoder! 
        - potremmo utilizzare un T-test, con  ipotesi che per ogni regione, le distribuzioni di dati MI - Rest sono differenti.
        - 68 * 250 = 17.000 é troppo! dobbiamo scegliere per bene le regioni! (aiutati con risultati di MLP su features a mano, forse?)
- Auto encoder evaluation: The average Pearson correlation coefficient between the reconstructed signals and the original signals for all 1-second segments in the test set was 0.939


### Brainstorming 5-12 Dec

- !!! Problema !!!
    ◊ Sto spingendo a studiare un criterio per la selezione delle regioni. Ma la selezione deve essere di GRUPPO o di SOGGETTO ??
- Se devo fare selezione di gruppo, come faccio? Raramente ho stesse regioni

- Domanda:<br>
    ◊ Quanto sono importanti le altre regioni per la struttura di una singola regione?<br>
        (stavo pensando che potrei addestrare un unico autoencoder sulla singola coppia EEG-MEG di una stessa regione)<br>
        (e sfruttare dati di tutti i pazienti e forse di tutte le regioni???)<br>
        (se sfrutto dati di tutte le regioni, posso assegnare un significato alla struttura dell'encoded layer, e poi quando addestro classificatore, posso usare conv layers!!!)<br>
            
            regioni
          0 1 2 3 4 5
        f 0 . . . . . .
        e 1 . . . . . .
        a 2 . . . . . .
        t 3 . . . . . .


        (o anche solo su sottogruppi di regioni che sono appartenenti a stessa zona del cervello??? (diminuisco features))
    ◊ Successivamente potrei selezionare regioni in base a T test su encoded layer! (Baseline vs MI) o verificare cosa succede prendendo tot reigoni o escludendo tot regioni

    
LOO Leave One Out
- Idea: potrei addestrare singolo autoencoder (PolarCAP setting) su tutte regioni, a singola regione (1 EEG. per volta) (tutti soggetti, o LOO) - INTERSUBJECT
- Poi potrei, per scelta di regioni, fatta su singolo sogetto, prendere le tot regioni che massimizzano la performance del soggetto 
- Cosa bella: Potrei addestrare su tutti i dati di una sessione precedente e poi FineTuning (frozen or not) su seesione successiva 

Varie pipelines:<br>
fare tutto prima su singolo soggetto tutte regioni (subj buono, il 2)<br>
windows di 0.5 sec, sovrapposizione di 0.25 sec,che indagano i trials da 3 a 6 secondi, dove devono fare il task<br>
(tirials x windows_per_trial x regions) = (96x12x68) = 70.000 samples
- autoencoder, very easy, su timeseries (forse anche CNN semplice, senza struttura autoencoder)
- invece di timeseries "raw", diverse bande (tipo VAEEG paper), "multiscale" CNN
- vedere se invece di timeseries i powerspectra
- IDEA SIMPATICA MA NON SO SE FATTIBILE: concatenare al bootleneck features estratte a mano, verificare con intepretability tools quale features sono più significative

### 08/01/2026

Come addestro autoencoder:		<br>
(0.5sec time series, 0.25s overlap, entire window 3s) =	<br>	
11 windows		<br>
96 trials		<br>
20 soggetti		<br>
68 regioni		<br>

| tipology           | region type       | specificity       | subjects        | N. of examples (factors) | N. of examples  | feasible (based on N. examples)?|
|--------------------|-------------------|-------------------|-----------------|---------------------------|----------------|- |
| 1 autoencoder      | single region     | region agnostic   | all subjects    | 11 x 96 x 20 x 68         | 1.436.160      |✓ |
| 1 autoencoder      | single region     | region agnostic   | single subject  | 11 x 96 x 1 x 68          | 71.808         |✓ |
| 68 autoencoders    | single region     | region specific   | all subjects    | 11 x 96 x 20 x 1          | 21.120         |- |
| 68 autoencoders    | single region     | region specific   | single subject  | 11 x 96 x 1 x 1           | 1.056          |- |
| 1 autoencoder      | multiple regions  | —                 | all subjects    | 11 x 96 x 20 x 1          | 21.120         |- |
| 1 autoencoder      | multiple regions  | —                 | single subject  | 11 x 96 x 1 x 1           | 1.056          |X |




Domanda:
- è necessario fisiologicamente scegliere le stesse regioni EEG e MEG?
    è possibile che una data attività cerebrale sia evidente in alcune regioni nell'EEG e in altre nella MEG?
    se devo scegliere stesse regioni, dobbiamo pensare a come fare
- Intensità del segnale è significativa (sospetto di si)
    in base a questo scegliere corretta normalizzazione

### Selezione unica di regioni a priori:
1. 
    - Scelgo un criterio di selezione (Cohen's d); 
    - Seleziono le regioni su singolo soggetto (Considero le prime TOT regioni per ogni soggetto) ; 
    - Seleziono le regioni selezionate per più soggetti
2. 
    - Scelgo un criterio di selezione (Cohen's d); 
    - Seleziono le regioni su singolo soggetto (Considero le prime TOT regioni per ogni soggetto) ; 
    - Seleziono le regioni che hanno posizione media (tra soggetti) più bassa

## 05 - 08 / 12 / 2025

- cerco di capire come visualizzare la robustezza di un method di selezione delle features al variare dei kfold
- grafico hist2d region vs rank in all folds (per ora sul criterio mio, ma dattabile a tutti i criteri tipo T-Test)
![BoG2](Ranking_Regions_basic.png)

- ho trovato un modo per fare il sort delle regioni, in modo da mettere in evidenza quelle più significative
![BoG](Ranking_Regions_sorted.png)


## 08 - 09/12/2025

- Un trovato un T test multivariato su pingouin (multivariate_ttest)
- Ho effettuato il T test: verifico l'ipotesi statistica per ogni soggetto, per ogni regione, per ogni fold, che la popolazione di Power specra di Rest abbia media diversa da MI (popolazioni approssimate da gaussiane)

## 10/12/2025

- automatizzo plot visualizzazione scelta regioni
- Confronto tra robustezza di T test e MyCriteria per selezionare regioni. MyCriteria é più Robusto tra i fold (trend coerente tra i soggetti)

## 11/12/2025

- Indago meglio su come si comportano i critieri di scelta delle regioni
- Noto che la distribuzione dei p-value su multi T test e abbastabza uniforme in scala log. variabilità da 10^-14 a 10^-1
- Alcune volte il MyCriteria scelglie di guardare frequenze troppo basse, (prima davo spettro da 2Hz in poi)
- Si risolve dando spettro in 4-30 Hz
- Rimane il problema che alcune volte focus su differenze grandi, ma non significative (ricadono all'inizio della banda di freq, vedi Img dopo, in questo caso selezionavo auesta ROI come prima) ◊ ATTENTO non significativa perchè ho notato che Rest ha varianza enorme! e anche perchè qui
![BoG2](PowerSpectra_sub5_ROI33.png)
- Ho visualizzato tutti i PowerSpectra delle differenti regioni del soggetto 4. Non ci sembra essere una differenza elevata tra MI e Rest. Come mai classifico bene?? Che sta succedendo??

## 12/12/2025


Ho fatto vedere a Marie
- visualizzazione robustezza (multi T test vs Mycriteria)
- corrispondenza con performances (no a priori regions)
- Power spectra regioni soggetto 4

Abbiamo osservato che:
- per il MyCriteria test sto facendo average su trials, average (selection) su frequency. ho molto più averaging
- per il multi T test su tutto il power spectra per come lo ho fatto, per ogni reigone ho "variabilità" su tanti samples e tanti trials
- il "molto più averaging" del MyCriteria, è molto probabile che sia questo a renderlo più robusto

- Fai t test su media su freq alpha e beta
- Vedere performances con questo nuovo multi (2 features) T test

- Ho fatto MyCriteria su 8 - 30 Hz. Stessa robustezza e Funiziona meglio effettivamente ! (ho controllato visivamente gli spectra delle regioni scelte e. sembra ok, migliore)
- Ho fatto multi(2 valori) T test su media su freq alpha e beta, ho notato che è più robusto rispetto multi T test su tutto vettore del power spectra, ma è sempore meno robusto del MyCriteria
- Oltre a robustezza, bisogna capire qunto sia buono un method di selezione. Visualizzo se sto selezionando regioni significative, ma non sempre riesco a dare interpretazione esaustiva. Provo a verificare bontà di selezione a posteriori, in base a quanto performano i miei classifier

In [3]:
96*11*68

71808

## 15/12/2025 Incontro con tutti


- Provo a verificare bontà di selezione a posteriori, in base a quanto performano i miei classifier.
- A questo scopo, check delle performance dei metodi di selezione "MyCriteria" e "T test su mean alpha beta", al variare di classifier, al variare di DataType.




Done:
- New criteria - effect size (verifica robustezza, performances, come per gli altri criteri)
- Verifica comunione di regioni tra soggetti tra le prime 20 regioni scelte (al variare dei criteri) (Images-Brain-Region_selection...)
- prova ad aumentare i kfold (RepeatedStratifiedKFold di scikit-learn)

## 17/12/2025


- testo il Choen_effect_size
- verifico la robustezza e scopro che sembra leggermente meno robusto sia del MyCriteria che del T test on mean alpha e beta
- verifico le performances e scopro che sono migliori (poco) sia del MyCriteria che del T test on mean alpha e beta

- Verifico se c'é sovrapposizione tra i soggetti nelle regioni scelte per cohen's d effect size.
- effettivamente (20 best regioni) molti scelgono quelle MOTORIE! (left hemi)

## 18/12/2025


Fatto (tutto contenuto nelle immagini BrainPlot):
- salva i plot dei brain (selection dei primi 20)
- controlla che succede con gli altri criteri
- controlla se almeno 1 delle regioni scelte tra le 20, per i soggetti, rientra in una di quelle motorie (left hemi)
- Controlla se la selezione la faccio calcolando gli spettri solamente dal secondo 3 in poi (Non sembra stabile quanto gli altri, e le performace sembrano peggio???)
- prova ad aumentare i kfold (RepeatedStratifiedKFold di scikit-learn)



## 19/12/2025 - Incontro con Marie

Done:
- MyCriteria: Dividi per il rest dello stesso bin di dove hai max(rest-MI) e vedi che succede (battezzato MyCriteria2)
- verify my effect size computation is the same of the pingoun one (pingouin effect size) (Paired=True) (verified it is Ok)
- verifica a quale frequenza corrisponde max (In Cohen's d Effect size, in order to have control on frequencies among folds, subjects etc)


## 23-29/12/2025

### 23/12/2025
- Verified my effect size computation is the same of the pingoun one (pingouin effect size) (Paired=True) (verified it is Ok, see Test notebook)
- Organized github folder 
- Migration on Salerno Workstation (environments, adaptation of codes, downloading data etc...)
- Compatibility between laptopts - Workstation (now I can work from every device)

### 24/12/2025
- Training with new version of MyCriteria selection (now not dividing for the max of rest, but for the value of the rest where i found the max(MI-Re))
- It is not better in performance than the normal MyCriteria
- Not more robusts

### 25/12/2025
- Sistemato il codice per fare i violin plot di tutti i soggetti al variare delle impostazioni


- Violin Plot - fixed classifier, method selection, varying Nrois, Datatype. 
- Observation: The more the ROIs, the better the performance (Sempre per EEG concat MEG) (SVC, RandomFoest; for each selection criterion; LDA solito comportamento strano)
- Bisogna capire fino a dove è significativa la differenza



### 26/12/2025
- provo ad incrementare i Kfold - 10 repeated stratified kfolds, "BCI_Performances_region_selection_on_repeated_10folds_8_30_Hz"
- Testato per MyCriteria, Cohen's d effect size, perogni classifier, per EEG, MEG, EEG concat MEG

### 27/12/2025
- Verifico che con 10 k-folds le performanes non sono variate
- grafico in maniera diversa, tenendo traccia di ciascun soggetto. 3 Violin plot per soggetto

### 28/12/2025
- verifico a quale bin di freq.z corrisponde il max di cohen's d effect size
- plot al variare dei soggetti, per le n_best regioni scelte

### 29/12/2025 - ...
- Lavoro sul paper di Mario; Inizio sistemando le figure; Trovo inconsistenze per NeuralNetworks

## 07/01/2026
Todo:
- Regions_Selection.ipynb -> Cohen's d -> Verify Frequency bin selection -> Controlla non solo dove è il massimo, ma potrei plottare la media (su regioni? Normalizzo?) di Cohen's d al variare del bin di frequenza
- fai un altro pannello di performance: Scelgo le 3 (5,10,...) migliori in "generale" su tutti i soggetti e vedo che succede alle performances in base a questo criterio di selezione generale, le stesse regioni per tutti i soggetti (controlla sempre se la 44, 32 etc sono scelte)
- append ai files di region selection le selezioni fatte senza dividere per kfold

## 08/01/2026 - 09/01/2026

Incontro con Marie<br>
Todo:
- Cerca di avere punti da discutere per incontro di lunedì
    - Recap di tutto quello fatto fino ad ora since ultimo incontro (choen, visualization on brains of region selection, kfolds DA FARE PLOT 10 vs 5 folds)
    - Mostra idee di diversi autoencoder. 
        - metodi principali: 1 autoencoder region agnostic (region selection 'a posteriori', fattibile da subito), 1 autoencoder multiple regions (region selection is important 'a priori')
    - Quali sono i parametri per cui mi servirebbe aiuto a fissare? (Window size, frequency filtered, overlap between windows, serve selezionare stesse regioni in MEG/EEG? (vedi brainstorming -> 08/01/2026) ...)


- Inizia a costruire pipeline più semplice per autoencoder a singolo soggetto, region agnostic (scegli soggetto 0 o 2 come iniziali)

- Se hai tempo, prova a vedere performance selezionando 3 regioni, al variare di tutte le combinazioni di 3 regioni, magari selezionare tra le regioni che performano meglio a singola regione (Visualization.ipynb -> ## Regions' Investigation ) 

Done:
- Criterio per selezione unica di regioni per tutti i soggetti (in vista di costruire un autoencoder a più regioni). 
    - Seleziona le TOT_1 regioni che vengono più scelte in generale per la selezione di singolo soggetto, e verifica che ci siano almeno le TOT_2 migliori reioni per soggetto considerate per tutti i soggetti.
    - O almeno grafica quante tra le migliori TOT_2 regioni di singolo soggetto sono selezionate tra le TOT_1 generali. 
    - Verifica poi quali sono le performances facendo una selezione unica per tutti i soggetti.
    - Dai uno sguardo se le regioni importanti left [4, 32, 44, 48] vengono incluse

- Working on global region selection (Regions_Selection.ipynb->Cohen->GlobalSelection)
    - found that (for Cohen based global selection) considering the best 15 for all subjects, and then taking the 10 most selected, we can retrieve also regions 32,44,48, both for MEG than EEG (poor region 4, it is not selected) 
    - the 10 most selected does not completely overlap between EEG and MEG! (what can I do?)

- Verifying performances on global selection (MyCriteria and Cohen): 
    saving in /Results/BCI_Performances_region_selection_on_repeated_10folds_8_30_Hz.pkl
- Comparison with subject-based selection (Visualization->Starting on 12 / 2025->Plot ALl subjects -> DIfferent Criteria). 
    Scopro che la scelta globale non è meglio.
    (LDA indifferente, SVC miglioramenti minimi per MyCriteria peggioramenti piccoli per Cohen, RandomForest peggioramenti Cohen EEG!)

## 10/01/2026
Todo:<br>
Trasporta i risultati a 10 folds sui file con 5 folds (global region selection)

# Access resources

- cluster ICM (I think Access available only from workstation desktop ??)
- workstation Dell office, ask Arthutr (VPN ? https://myicm.icm-institute.org/accessing-telecommuting-resources)